<div style="display:flex; align-items:center; gap:10px; margin-bottom:8px;">
  <span style="font-size:26px; color:#9558B2;">●</span>
  <span style="font-size:26px; color:#389826;">●</span>
  <span style="font-size:26px; color:#CB3C33;">●</span>
  <span style="font-size:26px; color:#4063D8;">●</span>
  <span style="font-size:30px; font-weight:700; margin-left:6px;">Julia</span>
</div>

# Julia с нуля — **Lesson 14**
## 📘 **Final Project / Capstone**
### **ThermalLab — численный анализатор процесса охлаждения**

**Cartesian School · Julia Course**  
**Автор:** Siergej Sobolewski  
**Copyright:** © 2026 Cartesian School


## Информация об уроке

| Поле | Значение |
|---|---|
| Курс | Julia с нуля |
| Номер урока | Lesson 14 |
| Тип | Final Project / Capstone |
| Уровень | Средний |
| Ориентировочное время | 6–10 часов |
| Требования | Lesson 0–13 |
| Основные области | функции, структуры данных, plotting, multiple dispatch, пакеты, производительность, линейная алгебра, численные методы |
| Результат | полноценная мини-система для анализа и моделирования |
| Автор | Siergej Sobolewski |
| Права | © 2026 Cartesian School |


## Цель Capstone

ThermalLab объединяет весь курс в одном проекте.

Система:

1. представляет физическую модель охлаждения;
2. хранит валидированные измерительные данные;
3. моделирует динамику системы;
4. поддерживает разные интеграторы через multiple dispatch;
5. оценивает параметр модели методом least squares;
6. вычисляет residuals и метрики ошибки;
7. выполняет benchmark реализаций;
8. при необходимости визуализирует данные;
9. экспортирует результаты;
10. содержит тесты.


## Связь со всем курсом

| Тема | Использование |
|---|---|
| Strings | отчёты и форматирование |
| Data Structures | `struct`, `Vector`, `NamedTuple` |
| Loops | симуляция |
| Conditionals | валидация |
| Functions | функциональная архитектура |
| Packages | `Pkg`, `LinearAlgebra`, `Statistics`, опционально `Plots` |
| Plotting | графики измерений и residuals |
| Multiple Dispatch | Euler / RK4 |
| Julia is Fast | timing и аллокации |
| Linear Algebra | least squares |
| Factorizations | оператор `\` |
| Numerical Computing | ODE, ошибка, валидация |


## Архитектура проекта

```text
ThermalLab
├── ThermalModel
├── MeasurementSet
├── SimulationResult
├── AbstractIntegrator
│   ├── EulerIntegrator
│   └── RK4Integrator
├── simulate(...)
├── estimate_k(...)
├── metrics(...)
├── benchmark
├── plotting
├── export
└── tests
```


# Часть I — окружение


## **1. Стандартные пакеты**


In [ ]:
using LinearAlgebra
using Statistics
using Random
using Printf
using Pkg

Random.seed!(2026)

println("Активный проект: ", Pkg.project().path)


## **2. Опциональный `Plots.jl`**


In [ ]:
const HAS_PLOTS = try
    @eval using Plots
    true
catch
    false
end

println("Plots.jl доступен: ", HAS_PLOTS)


### Важно

Notebook не выполняет автоматически `Pkg.add("Plots")`.

Если вам нужны графики:

```text
pkg> add Plots
```


# Часть II — физическая модель


## **3. Закон охлаждения Ньютона**

\[
\frac{dT}{dt} = -k(T-T_{env})
\]

Аналитическое решение:

\[
T(t)=T_{env}+(T_0-T_{env})e^{-kt}
\]


## **4. `ThermalModel`**


In [ ]:
struct ThermalModel{T<:Real}
    k::T
    ambient::T

    function ThermalModel(k::T, ambient::T) where {T<:Real}
        k > zero(T) || throw(ArgumentError("k должно быть положительным"))
        new{T}(k, ambient)
    end
end


In [ ]:
model = ThermalModel(0.12, 20.0)
@show model


## **5. `MeasurementSet`**


In [ ]:
struct MeasurementSet{T<:Real}
    time::Vector{T}
    temperature::Vector{T}

    function MeasurementSet(time::Vector{T}, temperature::Vector{T}) where {T<:Real}
        length(time) == length(temperature) ||
            throw(ArgumentError("time и temperature должны иметь одинаковую длину"))

        isempty(time) &&
            throw(ArgumentError("набор не может быть пустым"))

        issorted(time) ||
            throw(ArgumentError("время должно быть отсортировано"))

        new{T}(time, temperature)
    end
end


## **6. `SimulationResult`**


In [ ]:
struct SimulationResult{T<:Real}
    time::Vector{T}
    temperature::Vector{T}
    method::Symbol
end


### Анализ архитектуры

Мы разделяем:

- модель;
- данные;
- результат симуляции.

Благодаря этому каждый тип имеет одну ответственность.


# Часть III — функции модели


## **7. Уравнение состояния и точное решение**


In [ ]:
thermal_rhs(model::ThermalModel, t, T) =
    -model.k * (T - model.ambient)

exact_temperature(model::ThermalModel, T0, t) =
    model.ambient + (T0 - model.ambient) * exp(-model.k * t)


In [ ]:
@show thermal_rhs(model, 0.0, 90.0)
@show exact_temperature(model, 90.0, 10.0)


# Часть IV — multiple dispatch


## **8. Абстракция интегратора**


In [ ]:
abstract type AbstractIntegrator end

struct EulerIntegrator <: AbstractIntegrator end
struct RK4Integrator <: AbstractIntegrator end


## **9. `step` для Euler**


In [ ]:
function step(::EulerIntegrator, model::ThermalModel, t::Real, T::Real, h::Real)
    return T + h * thermal_rhs(model, t, T)
end


## **10. `step` для RK4**


In [ ]:
function step(::RK4Integrator, model::ThermalModel, t::Real, T::Real, h::Real)
    k1 = thermal_rhs(model, t, T)
    k2 = thermal_rhs(model, t + h/2, T + h*k1/2)
    k3 = thermal_rhs(model, t + h/2, T + h*k2/2)
    k4 = thermal_rhs(model, t + h, T + h*k3)

    return T + h * (k1 + 2k2 + 2k3 + k4) / 6
end


In [ ]:
@show methods(step)


### Почему multiple dispatch?

Мы не используем строковое условие вида `if method == "euler"`.

Движок выбирает подходящий метод `step` на основе типа интегратора.


# Часть V — движок симуляции


## **11. `simulate`**


In [ ]:
function simulate(
    model::ThermalModel,
    integrator::AbstractIntegrator,
    T0::Real,
    tspan::Tuple{<:Real,<:Real},
    h::Real,
)
    t0, t1 = tspan

    h > 0 || throw(ArgumentError("h должно быть положительным"))
    t1 > t0 || throw(ArgumentError("t1 должно быть больше t0"))

    n = Int(floor((t1 - t0) / h))

    times = Vector{Float64}(undef, n + 1)
    temperatures = Vector{Float64}(undef, n + 1)

    times[1] = float(t0)
    temperatures[1] = float(T0)

    for i in 1:n
        t = times[i]
        T = temperatures[i]

        times[i + 1] = t + h
        temperatures[i + 1] = step(integrator, model, t, T, h)
    end

    method = integrator isa EulerIntegrator ? :euler : :rk4

    return SimulationResult(times, temperatures, method)
end


In [ ]:
result_euler = simulate(model, EulerIntegrator(), 90.0, (0.0, 30.0), 0.1)
result_rk4   = simulate(model, RK4Integrator(),   90.0, (0.0, 30.0), 0.1)

@show result_euler.temperature[end]
@show result_rk4.temperature[end]


## **12. Точное решение для временного ряда**


In [ ]:
exact_series(model::ThermalModel, T0, times) =
    [exact_temperature(model, T0, t) for t in times]

exact_rk4 = exact_series(model, 90.0, result_rk4.time)


# Часть VI — метрики


## **13. Residuals, MAE, RMSE**


In [ ]:
residuals(predicted, observed) = predicted .- observed

mae(predicted, observed) =
    mean(abs.(residuals(predicted, observed)))

rmse(predicted, observed) =
    sqrt(mean(abs2, residuals(predicted, observed)))


In [ ]:
@show mae(result_euler.temperature, exact_rk4)
@show rmse(result_euler.temperature, exact_rk4)
@show mae(result_rk4.temperature, exact_rk4)
@show rmse(result_rk4.temperature, exact_rk4)


## **14. Проверка преимущества RK4**


In [ ]:
@assert rmse(result_rk4.temperature, exact_rk4) <
        rmse(result_euler.temperature, exact_rk4)

println("PASS — RK4 точнее Euler для данного шага.")


# Часть VII — измерительные данные


## **15. Генерация синтетических данных**


In [ ]:
function generate_measurements(
    model::ThermalModel,
    T0::Real,
    times::Vector{Float64};
    noise_std::Real=0.4,
    rng=Random.default_rng(),
)
    noise_std >= 0 ||
        throw(ArgumentError("noise_std не может быть отрицательным"))

    exact = exact_series(model, T0, times)
    noisy = exact .+ noise_std .* randn(rng, length(times))

    return MeasurementSet(copy(times), noisy)
end


In [ ]:
measurement_times = collect(0.0:1.0:30.0)

data = generate_measurements(
    model,
    90.0,
    measurement_times;
    noise_std=0.35,
)

@show length(data.time)
@show data.temperature[1:5]


# Часть VIII — линейная алгебра и least squares


## **16. Линеаризация модели**

\[
T(t)-T_{env}=(T_0-T_{env})e^{-kt}
\]

После логарифмирования:

\[
\ln(T(t)-T_{env}) = \beta_0 + \beta_1 t
\]

и:

\[
k=-\beta_1
\]


## **17. Матрица проекта**


In [ ]:
function prepare_linearized_problem(data::MeasurementSet, ambient::Real)
    delta = data.temperature .- ambient
    valid = delta .> 0

    t = data.time[valid]
    y = log.(delta[valid])

    X = hcat(ones(length(t)), t)

    return X, y
end

X, y = prepare_linearized_problem(data, model.ambient)

@show size(X)
@show length(y)


## **18. Least squares через `\`**


In [ ]:
β = X \ y

estimated_intercept = β[1]
estimated_slope = β[2]
estimated_k = -estimated_slope

@show β
@show estimated_k
@show model.k


### Важно

Мы не используем `inv(X'X) * X'y`.

Используем:

```julia
X \ y
```


## **19. Функция `estimate_k`**


In [ ]:
function estimate_k(data::MeasurementSet, ambient::Real)
    X, y = prepare_linearized_problem(data, ambient)
    β = X \ y
    k_est = -β[2]

    k_est > 0 ||
        throw(ArgumentError("оценённое k не является положительным"))

    return (
        k = k_est,
        intercept = β[1],
        coefficients = β,
        residual = X * β - y,
    )
end


In [ ]:
fit = estimate_k(data, model.ambient)

@show fit.k
@show norm(fit.residual)


# Часть IX — оценка модели с найденным параметром


## **20. Модель с оценённым параметром**


In [ ]:
estimated_model = ThermalModel(fit.k, model.ambient)

estimated_prediction = exact_series(
    estimated_model,
    data.temperature[1],
    data.time,
)

@show estimated_model


## **21. Метрики модели**


In [ ]:
function model_metrics(predicted, observed)
    r = residuals(predicted, observed)

    return (
        mae = mean(abs.(r)),
        rmse = sqrt(mean(abs2, r)),
        max_abs_error = maximum(abs.(r)),
        mean_residual = mean(r),
    )
end

metrics = model_metrics(estimated_prediction, data.temperature)

@show metrics


## **22. Коэффициент `R²`**


In [ ]:
function r_squared(predicted, observed)
    ss_res = sum(abs2, observed .- predicted)
    ss_tot = sum(abs2, observed .- mean(observed))

    ss_tot == 0 && return NaN
    return 1 - ss_res / ss_tot
end

R2 = r_squared(estimated_prediction, data.temperature)

@show R2


# Часть X — производительность


## **23. Warm-up**


In [ ]:
simulate(model, EulerIntegrator(), 90.0, (0.0, 30.0), 0.01)
simulate(model, RK4Integrator(),   90.0, (0.0, 30.0), 0.01)

println("Warm-up завершён.")


## **24. Простой benchmark**


In [ ]:
function best_elapsed(f; samples=10)
    best = Inf

    for _ in 1:samples
        elapsed = @elapsed f()
        best = min(best, elapsed)
    end

    return best
end

euler_time = best_elapsed(
    () -> simulate(model, EulerIntegrator(), 90.0, (0.0, 30.0), 0.01)
)

rk4_time = best_elapsed(
    () -> simulate(model, RK4Integrator(), 90.0, (0.0, 30.0), 0.01)
)

@show euler_time
@show rk4_time


## **25. Аллокации**


In [ ]:
euler_alloc = @allocated simulate(
    model, EulerIntegrator(), 90.0, (0.0, 30.0), 0.01
)

rk4_alloc = @allocated simulate(
    model, RK4Integrator(), 90.0, (0.0, 30.0), 0.01
)

@show euler_alloc
@show rk4_alloc


## **26. Точность и вычислительная стоимость**


In [ ]:
function terminal_error(result::SimulationResult, model, T0)
    exact = exact_temperature(model, T0, result.time[end])
    return abs(result.temperature[end] - exact)
end

e = simulate(model, EulerIntegrator(), 90.0, (0.0, 30.0), 0.05)
r = simulate(model, RK4Integrator(),   90.0, (0.0, 30.0), 0.05)

@show terminal_error(e, model, 90.0)
@show terminal_error(r, model, 90.0)


### Вывод

Алгоритм оценивается по:

- времени;
- памяти;
- точности;
- устойчивости.


# Часть XI — plotting


## **27. Измерения и аппроксимирующая модель**


In [ ]:
if HAS_PLOTS
    p = Plots.scatter(
        data.time,
        data.temperature;
        label="измерения",
        xlabel="Время",
        ylabel="Temperatura [°C]",
        title="ThermalLab — аппроксимация модели",
    )

    Plots.plot!(
        p,
        data.time,
        estimated_prediction;
        label="model",
        linewidth=2,
    )

    display(p)
else
    println("Plots.jl не установлен — график пропущен.")
end


## **28. Euler vs RK4 vs точное решение**


In [ ]:
if HAS_PLOTS
    h = 0.5

    e = simulate(model, EulerIntegrator(), 90.0, (0.0, 30.0), h)
    r = simulate(model, RK4Integrator(),   90.0, (0.0, 30.0), h)
    exact = exact_series(model, 90.0, e.time)

    p = Plots.plot(
        e.time,
        exact;
        label="точное",
        linewidth=3,
        xlabel="Время",
        ylabel="Temperatura [°C]",
        title="Сравнение интеграторов",
    )

    Plots.plot!(p, e.time, e.temperature; label="Euler")
    Plots.plot!(p, r.time, r.temperature; label="RK4")

    display(p)
else
    println("Plots.jl не установлен — график пропущен.")
end


## **29. Residuals**


In [ ]:
if HAS_PLOTS
    r = residuals(estimated_prediction, data.temperature)

    p = Plots.scatter(
        data.time,
        r;
        xlabel="Время",
        ylabel="Residual [°C]",
        title="Residuals модели",
        label="residual",
    )

    display(p)
else
    println("Plots.jl не установлен — график пропущен.")
end


# Часть XII — отчётность


## **30. Текстовый отчёт**


In [ ]:
function print_report(true_model, estimated_model, metrics, R2)
    println("="^60)
    println("THERMALLAB — ИТОГОВЫЙ ОТЧЁТ")
    println("="^60)

    @printf("k истинное:    %.6f\n", true_model.k)
    @printf("k оценённое:     %.6f\n", estimated_model.k)
    @printf("ошибка k:           %.6f\n", abs(true_model.k - estimated_model.k))
    @printf("MAE:               %.6f\n", metrics.mae)
    @printf("RMSE:              %.6f\n", metrics.rmse)
    @printf("MAX |error|:       %.6f\n", metrics.max_abs_error)
    @printf("mean residual:     %.6f\n", metrics.mean_residual)
    @printf("R²:                %.6f\n", R2)

    println("="^60)
end

print_report(model, estimated_model, metrics, R2)


## **31. Экспорт CSV**


In [ ]:
function export_csv(filename, time, measured, predicted)
    length(time) == length(measured) == length(predicted) ||
        throw(ArgumentError("серии должны иметь одинаковую длину"))

    open(filename, "w") do io
        println(io, "time,measured,predicted,residual")

        for i in eachindex(time, measured, predicted)
            r = predicted[i] - measured[i]

            println(
                io,
                time[i], ",",
                measured[i], ",",
                predicted[i], ",",
                r,
            )
        end
    end

    return filename
end


In [ ]:
csv_path = export_csv(
    "thermallab_results.csv",
    data.time,
    data.temperature,
    estimated_prediction,
)

println("Сохранено: ", csv_path)


# Часть XIII — тесты


## **32. Контрактные тесты**


In [ ]:
function run_capstone_tests()
    test_model = ThermalModel(0.1, 20.0)

    @assert exact_temperature(test_model, 100.0, 0.0) == 100.0

    e = simulate(
        test_model,
        EulerIntegrator(),
        100.0,
        (0.0, 10.0),
        0.1,
    )

    r = simulate(
        test_model,
        RK4Integrator(),
        100.0,
        (0.0, 10.0),
        0.1,
    )

    exact = exact_series(test_model, 100.0, r.time)

    @assert length(e.time) == length(e.temperature)
    @assert length(r.time) == length(r.temperature)
    @assert all(diff(e.time) .> 0)
    @assert all(diff(r.time) .> 0)

    @assert r.temperature[end] < r.temperature[1]
    @assert r.temperature[end] > test_model.ambient

    @assert rmse(r.temperature, exact) <
            rmse(e.temperature, exact)

    @assert model_metrics(exact, exact).rmse == 0.0
    @assert isapprox(r_squared(exact, exact), 1.0)

    return true
end

@assert run_capstone_tests()

println("PASS — все тесты Capstone завершены успешно.")


## **33. Тест валидации некорректных данных**


In [ ]:
validation_test = try
    ThermalModel(-0.5, 20.0)
    false
catch e
    e isa ArgumentError
end

@assert validation_test

println("PASS — валидация модели работает.")


# Часть XIV — анализ архитектуры


## **34. Модульность**

| Компонент | Ответственность |
|---|---|
| `ThermalModel` | параметры модели |
| `MeasurementSet` | измерительные данные |
| `SimulationResult` | результат интегрирования |
| `step` | один шаг алгоритма |
| `simulate` | выполнение симуляции |
| `estimate_k` | оценка параметра |
| `model_metrics` | оценка качества |
| `print_report` | представление результата |
| `export_csv` | сохранение данных |


## **35. Расширяемость через dispatch**

Чтобы добавить новый интегратор:

```julia
struct HeunIntegrator <: AbstractIntegrator end
```

и новый метод:

```julia
step(::HeunIntegrator, ...)
```

Изменять `simulate` не требуется.


# Часть XV — расширяющие задания


## **36. Extension A — Heun**

Добавьте `HeunIntegrator` и сравните Euler / Heun / RK4 для одинаковых шагов.


In [ ]:
# Twoja implementacja:


## **37. Extension B — оценка `T_env`**

Расширьте процедуру оценивания так, чтобы неизвестными одновременно были `k` и `T_env`.

Это приводит к нелинейной задаче оптимизации.


In [ ]:
# Определите целевую функцию:
# objective(params) = ...


## **38. Extension C — данные из файла**

Замените синтетические данные реальным файлом.

Возможные инструменты:

- `DelimitedFiles`;
- `CSV.jl`;
- `DataFrames.jl`.


## **39. Extension D — собственный пакет**

Перенесите проект в структуру:

```text
ThermalLab/
├── Project.toml
├── src/
│   └── ThermalLab.jl
├── test/
│   └── runtests.jl
└── README.md
```


# Часть XVI — критерии зачёта


## **40. Рубрика Capstone**

| Критерий | Баллы |
|---|---:|
| структуры данных | 10 |
| Euler | 8 |
| RK4 | 10 |
| multiple dispatch | 10 |
| симуляция | 10 |
| least squares | 12 |
| метрики | 8 |
| производительность | 8 |
| визуализация | 8 |
| тесты | 8 |
| документация и качество кода | 8 |
| **Итого** | **100** |


## **41. Минимальные требования для зачёта**

Проект считается зачтённым, если:

1. выполняется без ошибок;
2. Euler и RK4 работают;
3. RK4 даёт меньшую ошибку в тесте;
4. `estimate_k` возвращает положительное `k`;
5. least squares использует `\`;
6. присутствуют метрики качества;
7. тесты проходят;
8. код разделён на функции;
9. используется multiple dispatch;
10. результат интерпретирован.


# Часть XVII — итоговый checkpoint


## **42. Итоговые вопросы**

1. Почему модель, измерения и результат имеют отдельные типы?
2. Где используется multiple dispatch?
3. Почему `simulate` не должна знать имя конкретного алгоритма?
4. Что даёт параметризованный `ThermalModel{T}`?
5. Почему данные валидируются при создании объекта?
6. Как генерируются синтетические измерения?
7. Как задача оценки `k` превращается в линейную?
8. Почему используется `X \ y`?
9. Что измеряет RMSE?
10. Что измеряет `R²`?
11. Почему residuals важны?
12. Почему производительность нужно анализировать вместе с точностью?
13. Что измеряет `@allocated`?
14. Почему `Plots.jl` является опциональным?
15. Почему notebook не выполняет автоматически `Pkg.add`?
16. Как добавить третий интегратор?
17. Как перенести проект в пакет?
18. Какие тесты должны находиться в `runtests.jl`?


# Часть XVIII — завершение курса


## **43. Что вы освоили в Lesson 0–14**

После этого курса вы умеете:

- писать идиоматичный код на Julia;
- проектировать собственные типы;
- использовать функции и multiple dispatch;
- управлять пакетами и окружениями;
- визуализировать данные;
- измерять производительность;
- выполнять вычисления линейной алгебры;
- применять факторизации;
- реализовывать базовые численные методы;
- создавать полноценный проект от модели до отчёта.


## **44. Главный урок Capstone**

Профессиональный проект объединяет:

1. модель данных;
2. алгоритмы;
3. валидацию;
4. тесты;
5. анализ ошибок;
6. производительность;
7. воспроизводимое окружение;
8. понятное представление результатов.

ThermalLab демонстрирует именно такой подход к проектированию.


## **45. Что дальше?**

| Направление | Экосистема Julia |
|---|---|
| ODE / scientific computing | DifferentialEquations.jl |
| оптимизация | Optimization.jl, JuMP.jl |
| данные | DataFrames.jl, CSV.jl |
| статистика | StatsBase.jl, GLM.jl |
| ML | MLJ.jl, Flux.jl |
| symbolic | Symbolics.jl |
| GPU | CUDA.jl |
| визуализация | Makie.jl |
| создание пакетов | PkgTemplates.jl |


## **46. Final Challenge**

Создайте аналогичный проект для другой предметной области:

- зарядка конденсатора;
- модель батареи;
- затухающие колебания;
- биологическая популяция;
- измерительный сигнал;
- температура CPU.

Требования:

1. минимум два интегратора;
2. multiple dispatch;
3. оценка параметра;
4. линейная алгебра;
5. минимум две метрики;
6. benchmark;
7. график;
8. тесты;
9. итоговый отчёт.


## **47. Завершение**

Вы завершили курс **Julia с нуля — Cartesian School**.

Главный результат курса — умение пройти путь:

> **проблема → модель → код → вычисления → валидация → результат**

и создать решение, которое является:

- читаемым;
- тестируемым;
- расширяемым;
- воспроизводимым;
- учитывающим особенности численных вычислений.


## Источники для дальнейшего развития

- Julia Manual
- Julia Standard Library
- Pkg documentation
- LinearAlgebra
- Statistics
- Plots.jl
- DifferentialEquations.jl / SciML
- Julia Performance Tips
- Julia Package Development documentation


---

# **Конец курса**

**Cartesian School · Julia Course**  
**Lesson 14 — Final Project / Capstone**  
**Проект:** ThermalLab  
**Автор:** Siergej Sobolewski  
**Copyright:** © 2026 Cartesian School

[← Lesson 13 — Numerical Computing](Lesson_13_Numerical_Computing_Julia_Cartesian_School_RU.ipynb)  
[Оглавление](../README.ru.md)

**Learn Programming. Build Real Software. Master AI.**
